# Case Study 15 — CUDA Vector Addition, Kernel Optimization and AI Configuration

**Before running:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.
The whole notebook takes about 5 minutes.

Steps: build → device info → one fully-instrumented run → experiment sweep →
train AI model → charts/tables → AI-selected vs default configuration.

## 0. Check the GPU and get the code

In [ ]:
!nvidia-smi

In [ ]:
import os
if not os.path.exists('pc-project'):
    !git clone https://github.com/TejusDHegde/pc-project.git
%cd pc-project
!pip -q install -r requirements.txt

## 1. Build the CUDA program (compiled for this GPU's compute capability)

In [ ]:
cc = !nvidia-smi --query-gpu=compute_cap --format=csv,noheader
arch = 'sm_' + cc[0].strip().replace('.', '')
print('Compiling for', arch)
!make clean >/dev/null
!make ARCH=-arch={arch}

In [ ]:
!./bin/vector_add info

## 2. One run showing every step of the CUDA workflow
host allocation → CPU baseline → `cudaMalloc` → host-to-device copy → kernel → device-to-host copy → verification

In [ ]:
!./bin/vector_add run --n 16777216 --strategy naive --block 256

In [ ]:
# Try other configurations yourself:
!./bin/vector_add run --n 16777216 --strategy float4 --block 512
!./bin/vector_add run --n 16777216 --strategy gridstride --block 32 --grid 40 --pageable

## 3. Experiment sweep
33 vector sizes (2^10 … 2^26) × 48 configurations (3 strategies × 6 block sizes × grid sizes),
plus CPU sequential, CPU OpenMP with 1…all cores, pageable vs pinned transfers.
Use `--max-exp 22` for a faster run.

In [ ]:
!./bin/vector_add sweep --out results --max-exp 26

## 4. Train the AI model (predicts kernel time, recommends the fastest configuration)

In [ ]:
!python3 ai/train_model.py

## 5. Charts and result tables

In [ ]:
!python3 analysis/plots.py
from IPython.display import Image, Markdown, display
display(Markdown(open('results/RESULTS.md').read().split('## Figures')[0]))
for f in sorted(os.listdir('results/figures')):
    display(Image(filename='results/figures/' + f, width=700))

## 6. Use the AI model to configure the CUDA program

In [ ]:
!python3 ai/predict.py --n 10000000 --run

In [ ]:
# The CUDA program itself reads the AI's recommendation table:
!./bin/vector_add auto --n 5000000

## 7. Download all results

In [ ]:
!zip -qr results.zip results models
from google.colab import files
files.download('results.zip')